# Beam with varying EI and q #

In [ ]:
import sympy as sp
import numpy as np

w = sp.symbols('w', cls=sp.Function)
C1, C2 = sp.symbols('C1 C2')
x = sp.symbols('x')
L = 1

X_c = [0.1, 0.5, 0.9]

# f = 0
# for x_c in X_c:
#     f = sp.exp(-(x - x_c)**2) + f

# E = sp.simplify(f)
# E = sp.nsimplify(1)
# E = sp.nsimplify(sp.sin(x*(2*sp.pi)) + 2)
E = sp.nsimplify(-0.8333*x**2 + 1.25*x + 0.883333)
# E = sp.simplify(sp.exp(-x**2))
# E = sp.simplify(sp.exp(-(x-0.3)**2) + sp.exp(-(x-0.6)**2))
# E = sp.exp(-(x-0.3)**2) + sp.exp(-(x-0.6)**2)
# E = sp.exp(-(x-0.3)**2)
# q = sp.nsimplify(-500*w(x)+10+400*sp.diff(w(x),x,2))
# q = sp.nsimplify(500)
# q = sp.nsimplify( sp.DiracDelta(x-L))
F = 1.

sp.plotting.plot(E,(x,0,L))

diffeq = sp.Eq(sp.diff(E*sp.diff(w(x),x,1),x,1), 0)
display(diffeq)

w = sp.dsolve(diffeq)
w = w.rhs
display(w)

eps = sp.diff(w, x)
# kappa = sp.diff(phi, x)
# M = EI * kappa
# V = sp.diff(M, x)

eq1 = sp.Eq(w.subs(x , 0) , 0)
eq2 = sp.Eq(eps.subs(x , L) , F)
# eq2 = sp.Eq(phi.subs(x , 0) , 0)

sol = sp.solve((eq1,eq2),
               (C1 ,C2))
w_sol = w.subs(sol)
# M_sol = M.subs(sol)

sp.plot(w_sol,(x,0,L))
# sp.plot(M_sol,(x,0,L));

In [ ]:
import sympy as sp
import numpy as np

w = sp.symbols('w', cls=sp.Function)
C1, C2 = sp.symbols('C1 C2')
x = sp.symbols('x')
L = 1

X_c = [0.1, 0.5, 0.9]

# f = 0
# for x_c in X_c:
#     f = sp.exp(-(x - x_c)**2) + f

E = sp.nsimplify(-0.8333*x**2 + 1.25*x + 0.883333)
F = 1.

sp.plotting.plot(E,(x,0,L))

diffeq = sp.Eq(E*sp.diff(w(x),x,1), F)
display(diffeq)

w = sp.dsolve(diffeq)
w = w.rhs
display(w)

eps = sp.diff(w, x)
# kappa = sp.diff(phi, x)
# M = EI * kappa
# V = sp.diff(M, x)

# eq1 = sp.Eq(w.subs(x , 0) , 0)
# eq2 = sp.Eq(eps.subs(x , L) , F)
# eq2 = sp.Eq(phi.subs(x , 0) , 0)

sol = sp.solve((eq2),
               (C1))
w_sol = w.subs(sol)
# M_sol = M.subs(sol)

sp.plot(w_sol,(x,0,L))
# sp.plot(M_sol,(x,0,L));

In [ ]:
def chebyshev_expansion(f, N):
    x = sp.symbols('x')
    # Transform x in [0, 1] to x' in [-1, 1]
    xp = 2 * x - 1

    # Create Chebyshev polynomials T_n(x')
    T = [sp.chebyshevt(n, xp) for n in range(N+1)]

    # Define the integration variable on the transformed interval
    x_transformed = sp.Symbol('x_transformed')
    f_transformed = f.subs(x, (x_transformed + 1) / 2)

    # Calculate coefficients for the expansion
    c = []
    for n in range(N+1):
        # Substitute x' back to x_transformed for integration
        integrand = f_transformed * T[n].subs(xp, x_transformed) * sp.sqrt(1 - x_transformed**2)
        # Adjust the integral limits for the weighting function
        cn = sp.integrate(integrand, (x_transformed, -1, 1))
        if n == 0:
            cn *= 1/sp.pi
        else:
            cn *= 2/sp.pi
        c.append(cn)

    # Construct the polynomial expansion
    chebyshev_expansion = sum(c[n] * T[n] for n in range(N+1))
    return chebyshev_expansion

# Example function to expand
f = sp.exp(x)  # e^x defined on [0, 1]

# Compute expansion to degree 3
expansion = chebyshev_expansion(f, 3)
print(sp.simplify(expansion))

In [ ]:
import numpy as np
import scipy.integrate as integrate
from numpy.polynomial.chebyshev import Chebyshev, chebval

def compute_chebyshev_coefficients(func, degree, a, b):
    # Map function to [-1, 1]
    def mapped_func(x):
        # Scale x from [-1, 1] to [a, b]
        return func((x + 1) * (b - a) / 2 + a)

    # Initialize coefficients
    coeffs = np.zeros(degree + 1)
    
    # Integrate to find each coefficient
    for n in range(degree + 1):
        # Chebyshev polynomial of the first kind
        Tn = Chebyshev.basis(n)
        
        # Define the integrand as the product of the mapped function and the nth Chebyshev polynomial
        integrand = lambda x: mapped_func(x) * Tn(x) / np.sqrt(1 - x**2)
        
        # Numerically integrate to find the nth coefficient
        integral, error = integrate.quad(integrand, -1, 1)
        
        # Factor adjustment based on the orthogonality relation
        if n == 0:
            coeffs[n] = integral / np.pi
        else:
            coeffs[n] = integral * 2 / np.pi
    
    return coeffs

def my_sin(x):
    # return 0.5*np.sin(1.2*np.pi*x) * x**2 + 0.5
    return np.sin(2.0*np.pi*x) + np.cos(3.0*np.pi*x) + np.sin(5.0*np.pi*x) + np.cos(12*np.pi*x)
    # return x - 0.5
    # return np.exp(x)
# Define the original function f(x) = exp(x) over [0, 1]
# func = np.exp
func = my_sin

# Number of coefficients (degree of the polynomial)
degree = 40
# degree = 1

# Compute coefficients
coeffs = compute_chebyshev_coefficients(func, degree, 0, 1)

# Print coefficients
print("Coefficients of the Chebyshev expansion:", coeffs)

# Optionally, evaluate the Chebyshev polynomial at points
x_values = np.linspace(0, 1, 400)
# Convert x_values to [-1, 1]
x_transformed = 2 * x_values - 1
# Evaluate the polynomial
approx_values = chebval(x_transformed, coeffs)

# Plotting, if needed
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(x_values, func(x_values), label='Original Function $e^x$', color='blue')
plt.plot(x_values, approx_values, label='Chebyshev Approximation', linestyle='--', color='red')
plt.title('Comparison of $e^x$ and its Chebyshev Approximation')
plt.xlabel('x')
plt.ylabel('Function values')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
x = np.array([0.2, 0.5, 0.8])
y = np.array([1.1, 1.3, 1.35])

# Fit a polynomial of degree 2 (quadratic)
coefficients = np.polyfit(x, y, 2)

print(coefficients)

# Create a polynomial function using the coefficients
p = np.poly1d(coefficients)

# Generate x values for plotting the polynomial
x_line = np.linspace(min(x) - 1, max(x) + 1, 100)
y_line = p(x_line)

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(x_line, y_line, label='Fitted Polynomial', color='red')
plt.scatter(x, y, color='blue', label='Data Points')
plt.title('Polynomial Fit to Three Points')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from scipy.special import eval_chebyt

# Parameters for the Gaussian Random Field
mean = 0
std = 1
lengthscale = 0.5
a, b = -1, 1

# def x_tr(x):
#     # Scale x from [-1, 1] to [a, b]
#     return ((x + 1) * (b - a) / 2 + a)
def x_tr(x):
    return (b - a) * x + a
        
# Define the squared exponential kernel function
def squared_exponential_kernel(x, y, sigma=std, l=lengthscale):
    return sigma**2 * np.exp(-((x - y)**2) / (2 * l**2))

def weight(x):
    return 1 / np.sqrt(1 - x**2)

# Number of Chebyshev polynomials
N = 8

# Compute the coefficients for the Chebyshev polynomials
def compute_coefficients(n, m):
        
    def integrand(x, y):
        # Evaluate the n-th and m-th Chebyshev polynomial at x
        T_n = eval_chebyt(n, x_tr(x))
        T_m = eval_chebyt(m, x_tr(y))
        # Integral of the kernel and the n-th Chebyshev polynomial
        return squared_exponential_kernel(x_tr(x), x_tr(y)) * T_n * T_m * weight(x_tr(x)) * weight(x_tr(y))

    # We use numerical double integration over the interval [0, 1] for both x and y
    result, error = integrate.dblquad(integrand, 0, 1, 0, 1)
    # print(f"  value: {result:.4e}; error: {error:.4e}")
    return result

# Calculate coefficients
coefficients = np.zeros((N+1,N+1))
for i in range(N+1):
    for j in range(i,N+1):
        print(f"Computing coefficient ({i},{j})")

        # all products of even and odd polynomials are 0
        if (i%2==1) != (j%2==1):
            continue
            
        result = compute_coefficients(i,j)

        # normalize coefficients 
        if i==0:
            result = result * 2 / np.pi
        else:
            result = result * 4 / np.pi

        if j==0:
            result = result * 2 / np.pi
        else:
            result = result * 4 / np.pi
            
        coefficients[i,j] = coefficients[j,i] = result
        

# Display the coefficients
# print("Chebyshev coefficients:\n", coefficients)

In [ ]:
from scipy.special import eval_chebyt

# Parameters for the Gaussian Random Field
mean = 0
std = 1
lengthscale = 0.5
a, b = -1, 1

# def x_tr(x):
#     # Scale x from [-1, 1] to [a, b]
#     return ((x + 1) * (b - a) / 2 + a)
def x_tr(x):
    return (b - a) * x + a
        
# Define the squared exponential kernel function
def squared_exponential_kernel(x, y, sigma=std, l=lengthscale):
    return sigma**2 * np.exp(-((x - y)**2) / (2 * l**2))

def weight(x):
    return 1 / np.sqrt(1 - x**2)

# Number of Chebyshev polynomials
N = 3

# Compute the coefficients for the Chebyshev polynomials
def compute_coefficients(kernel, degree, interval):
        
    def integrand(x, y, n, m):
        # Evaluate the n-th and m-th Chebyshev polynomial at x
        T_n = eval_chebyt(n, x_tr(x))
        T_m = eval_chebyt(m, x_tr(y))
        # Integral of the kernel and the n-th Chebyshev polynomial
        return kernel(x_tr(x), x_tr(y)) * T_n * T_m * weight(x_tr(x)) * weight(x_tr(y))

    
    # Calculate coefficients
    coefficients = np.zeros((degree+1,degree+1))
    for i in range(degree+1):
        for j in range(i,degree+1):
            print(f"Computing coefficient ({i},{j})")
    
            # all products of even and odd polynomials are 0
            if (i%2==1) != (j%2==1):
                continue
                
            # result = compute_coefficients(i,j)
            result, _ = integrate.dblquad(lambda x, y: integrand(x, y, i, j),
                                          interval[0], interval[1], interval[0], interval[1])
    
            # normalize coefficients 
            if i==0:
                result = result * 2 / np.pi
            else:
                result = result * 4 / np.pi
    
            if j==0:
                result = result * 2 / np.pi
            else:
                result = result * 4 / np.pi
                
            coefficients[i,j] = coefficients[j,i] = result
    
    return coefficients

coefficients = compute_coefficients(squared_exponential_kernel, N, (0,1))

In [ ]:
x = np.linspace(0,1,100)
x_transformed = 2 * x - 1
Phi = np.array([eval_chebyt(i, x_transformed) for i in range(N+1)])
cov = Phi.T @ coefficients @ Phi

fig, ax = plt.subplots()
# ax.plot(x, np.sqrt(np.diag(cov)))
ax.plot(x, np.diag(cov))

In [ ]:
rng = np.random.default_rng(0)

x = np.linspace(0,1,100)
x_transformed = 2 * x - 1
Phi = np.array([eval_chebyt(i, x_transformed) for i in range(N+1)])

fig, ax = plt.subplots()
n_samples = 100

for i in range(n_samples):
    y = Phi.T @ rng.multivariate_normal(np.zeros(coefficients.shape[0]), coefficients)
    ax.plot(y, c='C0', alpha=0.5)

In [ ]:
import sympy as sp
import numpy as np

w = sp.symbols('w', cls=sp.Function)
C1, C2 = sp.symbols('C1 C2')
x = sp.symbols('x')
L = 1
mean = 5.0

rng = np.random.default_rng()
coeff_real = rng.multivariate_normal(np.zeros(N+1), coefficients)
print(coeff_real)

# f = 0
# for x_c in X_c:
#     f = sp.exp(-(x - x_c)**2) + f

E = mean

for i in range(N+1):
    E += sp.nsimplify(np.round(coeff_real[i], 2) * sp.chebyshevt(i, x))

E = sp.nsimplify(E)

# print(E)

# E = sp.nsimplify(-0.8333*x**2 + 1.25*x + 0.883333)
F = 1.

sp.plotting.plot(E,(x,0,L))

diffeq = sp.Eq(E*sp.diff(w(x),x,1), F)
display(diffeq)

w = sp.dsolve(diffeq)
w = w.rhs
display(w)

eps = sp.diff(w, x)
# kappa = sp.diff(phi, x)
# M = EI * kappa
# V = sp.diff(M, x)

# eq1 = sp.Eq(w.subs(x , 0) , 0)
# eq2 = sp.Eq(eps.subs(x , L) , F)
# eq2 = sp.Eq(phi.subs(x , 0) , 0)

sol = sp.solve((eq2),
               (C1))
w_sol = w.subs(sol)
# M_sol = M.subs(sol)

sp.plot(w,(x,0.,L))
# sp.plot(M_sol,(x,0,L));

In [ ]:
import sympy as sp
import numpy as np

w = sp.symbols('w', cls=sp.Function)
C1, C2 = sp.symbols('C1 C2')
x = sp.symbols('x')
L = 1
mean = 5.0

rng = np.random.default_rng()
F = 1.
a = 2
b = 4

E = 1+ sp.Heaviside(x, a) + sp.Heaviside(x-0.5, -a+b)

sp.plotting.plot(E,(x,0,L))

diffeq = sp.Eq(E*sp.diff(w(x),x,1), F)
display(diffeq)

w = sp.dsolve(diffeq)
w = w.rhs
display(w)

init_cond = sp.Eq(w.subs(x, 0), 0)

sol = sp.solve((init_cond),
               (C1))
w_sol = w.subs(sol)

sp.plot(w_sol,(x,0,L))

In [ ]:
import sympy as sp
import numpy as np

# Define the symbols
w = sp.symbols('w', cls=sp.Function)
C1 = sp.symbols('C1')
x = sp.symbols('x')
a, b, c, d = sp.symbols('a, b, c, d')

# Define the problem parameters
L = 1

# Define the expression E
E = a*sp.Heaviside(x) + b*sp.Heaviside(x - 0.25) + c*sp.Heaviside(x - 0.5) + d*sp.Heaviside(x-0.75)
E = E.rewrite(sp.Piecewise)
display(E)
# display(E.subs((a,b),(1,2)))
# E = 1.0
# sp.plotting.plot(E.subs((a,b), (1, 2)),(x,0,L))

# Define the ODE
diffeq = sp.Eq(E * sp.diff(w(x), x), 1.0)  # Assuming F=1 as in your script
display(diffeq)

# Solve the differential equation
sol = sp.dsolve(diffeq)
w = sol.rhs
display(w)

# Define initial conditions (example: w(0) = 0)
init_cond = sp.Eq(w.subs(x, 0), 0)

# Solve for constants
constants = sp.solve(init_cond, C1)
if isinstance(constants, dict):
    w_sol = w.subs(constants)
else:
    w_sol = w.subs(C1, constants[0])

# Substitute specific values for a and b in the solution
a_val = 1
b_val = 2
c_val = 3
d_val = 5
w_sol_substituted = w_sol.subs({a: a_val, b: b_val, c: c_val, d:d_val})
display(sp.diff(w_sol, a))

# Plot the solution
sp.plot(w_sol_substituted, (x, 0, 1), title='Solution of the ODE', ylabel='w(x)')

In [ ]:
rng = np.random.default_rng(0)
rng.random(size=n_params)

In [ ]:
import sympy as sp
import numpy as np

rng = np.random.default_rng(0)

n_params = 7

# Define the symbols
w = sp.symbols('w', cls=sp.Function)
C1 = sp.symbols('C1')
x = sp.symbols('x')
params = sp.symbols([f'p_{str(i)}' for i in range(n_params)])
offset = 1.

# Define the problem parameters
L = 1
# E = offset + (params[0] - offset) * sp.Heaviside(x)
E = sp.simplify('0')
E = sp.S('0')
for i in range(n_params):
    # E += params[i] * sy.Piecewise((0, x < (i / n_params)),
    #                                         (0.5, x == ((i+1) / n_params)),
    #                                         (1, True))V
    # E += (params[i + 1] - params[i]) * sp.Heaviside(x - (i + 1) / n_params)
    
    E += (params[i]) * sp.Heaviside(x - i / n_params)
    
E = E.rewrite(sp.Piecewise)
print(f"Young's modulus:\n{E}\n")

# # Define the expression E
# E = 1 + a*sp.Heaviside(x) + b*sp.Heaviside(x - 0.25) + c*sp.Heaviside(x - 0.5) + d*sp.Heaviside(x-0.75)
# E = E.rewrite(sp.Piecewise)
display(E)
# display(E.subs((a,b),(1,2)))
# E = 1.0
# sp.plotting.plot(E.subs((a,b), (1, 2)),(x,0,L))

# Define the ODE
diffeq = sp.Eq(E * sp.diff(w(x), x), 1.0)  # Assuming F=1 as in your script
display(diffeq)

# Solve the differential equation
sol = sp.dsolve(diffeq)
w = sol.rhs
display(w)

# Define initial conditions (example: w(0) = 0)
init_cond = sp.Eq(w.subs(x, 0), 0)

# Solve for constants
constants = sp.solve(init_cond, C1)
if isinstance(constants, dict):
    w_sol = w.subs(constants)
else:
    w_sol = w.subs(C1, constants[0])

# Substitute specific values for a and b in the solution
vals = rng.random(size=n_params)
w_sol_substituted = w_sol.subs(zip(params, vals))
display(sp.diff(w_sol, 'p_0'))

print(f"\nvals: {vals}\n")
# Plot the solution
sp.plot(w_sol_substituted, (x, 0, 1), title='Solution of the ODE', ylabel='w(x)')

In [ ]:
init_cond

In [ ]:
np.linspace(0,1,n_params + 1)

In [ ]:
import sympy as sp
import numpy as np

rng = np.random.default_rng(0)
n_params = 4

# Define the symbols
w = sp.symbols('w', cls=sp.Function)
C1 = sp.symbols('C1')
x = sp.symbols('x')
params = sp.symbols([f'p_{str(i)}' for i in range(n_params)])
# intervals = sp.symbols([f'b_{str(i)}' for i in range(n_params)])
# # intervals.insert(0, sp.symbols('b_0'))
# intervals_vals = np.linspace(0, 1, n_params)
offset = 1.

# Define the problem parameters
L = 1
E = offset + (params[0] - offset) * sp.Heaviside(x)
# E = sp.simplify('0')
# E = sp.S('0')
for i in range(n_params -1):
    # E += params[i] * sy.Piecewise((0, x < (i / n_params)),
    #                                         (0.5, x == ((i+1) / n_params)),
    #                                         (1, True))V
    E += (params[i + 1] - params[i]) * sp.Heaviside(x - (i+1) / n_params)
    
    # E += (params[i]) * sp.Heaviside(x - i / n_params)
    
E = E.rewrite(sp.Piecewise)
print(f"Young's modulus:\n{E}\n")

# # Define the expression E
# E = 1 + a*sp.Heaviside(x) + b*sp.Heaviside(x - 0.25) + c*sp.Heaviside(x - 0.5) + d*sp.Heaviside(x-0.75)
# E = E.rewrite(sp.Piecewise)
display(E)
# display(E.subs((a,b),(1,2)))
# E = 1.0
# sp.plotting.plot(E.subs((a,b), (1, 2)),(x,0,L))

# Define the ODE
diffeq = sp.Eq(E * sp.diff(w(x), x), 1.0)  # Assuming F=1 as in your script
display(diffeq)

# Solve the differential equation
sol = sp.dsolve(diffeq)
w = sol.rhs
display(w)

# Define initial conditions (example: w(0) = 0)
init_cond = sp.Eq(w.subs(x, 0), 0)

# Solve for constants
constants = sp.solve(init_cond, C1)
if isinstance(constants, dict):
    w_sol = w.subs(constants)
else:
    w_sol = w.subs(C1, constants[0])

# Substitute specific values for a and b in the solution
vals = 3 + 3*rng.random(size=n_params)
w_sol_substituted = w_sol.subs(zip(params, vals))
# w_sol_substituted = w_sol_substituted.subs(zip(inverals, intervals_vals))
display(sp.diff(w_sol, 'p_0'))

print(f"\nvals: {vals}\n")

# Plot the solution
sp.plot(w_sol_substituted, (x, 0, 1), title='Solution of the ODE', ylabel='w(x)')

In [ ]:
n_params = 11
# u = sp.symbols('w', cls=sp.Function)
x = sp.symbols('x')
params = sp.symbols([f'p_{str(i)}' for i in range(n_params)])
u_i = []
conds = []

for i in range(n_params):
    # u += (x - (i/n_params)) / params[i]
    u_i.append(sp.S((x - (i/n_params)) / params[i]))
    conds.append(sp.S(x < (i+1) / n_params))
    for j in range(i):
        u_i[i] += (1/n_params) / params[j]
    # display(u_i[i])

conds[-1] = sp.S('1')

u = sp.Piecewise(*zip(u_i, conds))
du = sp.diff(u, params[0])
ddu = sp.diff(du, params[0])
# display(ddu)

u_np = sp.lambdify((x, *params), u)

import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
params = 1 + 3 * rng.random(n_params)

x = np.linspace(0,1,100)

y = u_np(x, *params)

fig, ax = plt.subplots()
ax.plot(x, y)